# Task 2.2 — Silver: Learning Sessions
## Notebook: 05_silver_learning_sessions

**Layer:** Silver (computed from Bronze LMS events)

**What this notebook does:**
- Reads `bronze.lms_events_bronze`
- Uses LAG window to compute time gap between consecutive events per student
- Flags gaps > 30 minutes as the start of a new session (is_new_session = 1)
- Cumulative SUM over is_new_session → unique session_id per student
- Aggregates per session: session_duration_mins, events_in_session,
  modules_touched, primary_activity
- Flags sessions longer than 8 hours as is_long_session_flag = true (NOT dropped)
- Writes to silver.learning_sessions — one row per session

**Source table:** mini_project_grp6.bronze.lms_events_bronze
**Target table:** mini_project_grp6.silver.learning_sessions

**Business Rules enforced:**
- Sessions > 8 hours are flagged, never dropped (data error marker)
- student_id and course_id must not be NULL
- session_duration_mins must be >= 0
- session_id must be unique per (student_id, session_id) combination

**Run order:** Run all cells top-to-bottom. Safe to re-run (OVERWRITE mode).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema names ---
CATALOG = "mini_project_grp6"
BRONZE  = "bronze"
SILVER  = "silver"

# --- Source and target table names (fully qualified) ---
LMS_BRONZE_TABLE     = f"{CATALOG}.{BRONZE}.lms_events_bronze"
SESSIONS_SILVER_TABLE = f"{CATALOG}.{SILVER}.learning_sessions"
DQ_AUDIT_TABLE       = f"{CATALOG}.audit.dq_audit"

# --- Session definition threshold ---
SESSION_GAP_MINUTES   = 30     # gap > 30 min between events = new session
LONG_SESSION_HOURS    = 8      # sessions > 8 hours are flagged as data errors
LONG_SESSION_MINUTES  = LONG_SESSION_HOURS * 60  # 480 minutes

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Set default catalog + schema ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(SILVER)

print(f"✅ Catalog  : {CATALOG}")
print(f"✅ Schema   : {SILVER}")
print(f"✅ Source   : {LMS_BRONZE_TABLE}")
print(f"✅ Target   : {SESSIONS_SILVER_TABLE}")
print(f"✅ Session gap threshold : {SESSION_GAP_MINUTES} minutes")
print(f"✅ Long session threshold: {LONG_SESSION_MINUTES} minutes ({LONG_SESSION_HOURS}h)")

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType,
    TimestampType, DoubleType, BooleanType
)
from delta.tables import DeltaTable

print("✅ Imports successful")

## Part A — Read Bronze + Source Validation

Read lms_events_bronze and validate before computing sessions.
We need at least 2 events per student for meaningful session detection.

In [0]:
# ============================================================
# CELL 5 — Read lms_events_bronze and validate source data
# ============================================================

lms_df = spark.table(LMS_BRONZE_TABLE)

total_rows      = lms_df.count()
null_student    = lms_df.filter(F.col("student_id").isNull()).count()
null_event_ts   = lms_df.filter(F.col("event_ts").isNull()).count()
distinct_students = lms_df.select("student_id").distinct().count()
distinct_events = lms_df.select("event_type").distinct().count()

print("── Source validation: lms_events_bronze ──────────")
print(f"   Total rows        : {total_rows:,}")
print(f"   Distinct students : {distinct_students:,}")
print(f"   Distinct event types: {distinct_events}")
print(f"   NULL student_id   : {null_student}")
print(f"   NULL event_ts     : {null_event_ts}")
print()

print("── event_type distribution ───────────────────────")
lms_df.groupBy("event_type").count().orderBy(F.desc("count")).show(truncate=False)

# Filter out rows with NULL student_id or NULL event_ts before session computation
# These cannot be assigned to a session meaningfully
lms_clean_df = lms_df.filter(
    F.col("student_id").isNotNull() &
    F.col("event_ts").isNotNull()
)

clean_count = lms_clean_df.count()
dropped     = total_rows - clean_count
print(f"ℹ  Rows after NULL filter : {clean_count:,}  (dropped {dropped} rows with NULL student_id or event_ts)")

## Part B — Session Detection with LAG Window

Session definition (from Task 2.2 spec):
A session is a consecutive sequence of events from the same student
where NO gap between two adjacent events exceeds 30 minutes.

Algorithm:
1. Order all events per student by event_ts using a Window function
2. Use LAG() to get the previous event_ts for each event (within same student)
3. Compute gap_seconds = current event_ts - previous event_ts
4. If gap_seconds > 1800 (30 min) OR previous event_ts is NULL (first event) → is_new_session = 1
5. Cumulative SUM of is_new_session over the student's ordered events → session_seq
6. session_id = student_id + "_" + session_seq (globally unique per student)

In [0]:
# ============================================================
# CELL 7 — Step 1: LAG window to detect session boundaries
#
# Window is partitioned by student_id and ordered by event_ts.
# LAG(event_ts, 1) gives the previous event timestamp for the same student.
# If the gap exceeds SESSION_GAP_MINUTES or the previous ts is NULL
# (first event ever for this student), we flag is_new_session = 1.
# ============================================================

# Define the window: per student, ordered by event_ts ascending
student_event_window = (
    Window
    .partitionBy("student_id")
    .orderBy("event_ts")
)

# Apply LAG and compute gap
lms_with_gap_df = (
    lms_clean_df
    # LAG: previous event_ts for same student
    .withColumn(
        "prev_event_ts",
        F.lag("event_ts", 1).over(student_event_window)
    )
    # Gap in seconds between this event and the previous one
    .withColumn(
        "gap_seconds",
        F.when(
            F.col("prev_event_ts").isNotNull(),
            (F.unix_timestamp("event_ts") - F.unix_timestamp("prev_event_ts"))
        ).otherwise(F.lit(None).cast(LongType()))
    )
    # is_new_session = 1 if this is the first event (NULL prev_ts) OR gap > threshold
    .withColumn(
        "is_new_session",
        F.when(
            F.col("prev_event_ts").isNull() |
            (F.col("gap_seconds") > (SESSION_GAP_MINUTES * 60)),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)

print("── Gap detection result ──────────────────────────")
print(f"   Total events processed  : {lms_with_gap_df.count():,}")
print()

print("── is_new_session distribution ───────────────────")
lms_with_gap_df.groupBy("is_new_session").count().show()

print("── Gap stats (seconds) for cross-session boundaries ──")
lms_with_gap_df.filter(
    F.col("gap_seconds").isNotNull()
).select(
    F.min("gap_seconds").alias("min_gap_sec"),
    F.max("gap_seconds").alias("max_gap_sec"),
    F.round(F.avg("gap_seconds"), 0).alias("avg_gap_sec"),
    F.percentile_approx("gap_seconds", 0.5).alias("median_gap_sec")
).show()

In [0]:
# ============================================================
# CELL 8 — Step 2: Cumulative SUM of is_new_session
#          → session_seq (sequential session number per student)
#          → session_id  (globally unique: student_id + "_S_" + session_seq)
#
# Cumulative SUM using a Window with rowsBetween(unboundedPreceding, currentRow)
# ensures each event in session N gets the same session_seq = N.
#
# Example for student STU000001:
#   event 1 → is_new_session=1 → cumsum=1 → session_seq=1
#   event 2 → is_new_session=0 → cumsum=1 → session_seq=1 (same session)
#   event 3 → is_new_session=0 → cumsum=1 → session_seq=1
#   event 4 → is_new_session=1 → cumsum=2 → session_seq=2 (new session after 30+ min gap)
# ============================================================

# Cumulative sum window: same student, ordered by event_ts, all preceding rows
cumsum_window = (
    Window
    .partitionBy("student_id")
    .orderBy("event_ts")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

lms_with_session_df = (
    lms_with_gap_df
    # session_seq = running count of session starts for this student
    .withColumn(
        "session_seq",
        F.sum("is_new_session").over(cumsum_window)
    )
    # session_id = globally unique per student-session combination
    .withColumn(
        "session_id",
        F.concat_ws("_S_", F.col("student_id"), F.col("session_seq").cast("string"))
    )
)

total_sessions = lms_with_session_df.select("session_id").distinct().count()
total_students = lms_with_session_df.select("student_id").distinct().count()
avg_sessions   = round(total_sessions / total_students, 2) if total_students > 0 else 0

print("── Session ID assignment complete ────────────────")
print(f"   Total events   : {lms_with_session_df.count():,}")
print(f"   Total sessions : {total_sessions:,}")
print(f"   Total students : {total_students:,}")
print(f"   Avg sessions per student: {avg_sessions}")
print()

print("── Sample: session_id assignment for one student ─")
sample_student = lms_with_session_df.select("student_id").first()[0]
lms_with_session_df.filter(
    F.col("student_id") == sample_student
).select(
    "student_id", "event_ts", "event_type",
    "gap_seconds", "is_new_session", "session_seq", "session_id"
).orderBy("event_ts").show(15, truncate=False)

## Part C — Compute Session-Level Aggregates

Now that every event has a session_id, we GROUP BY session_id to compute:

- session_start_ts     : MIN(event_ts) in the session
- session_end_ts       : MAX(event_ts) in the session
- session_duration_mins: (session_end - session_start) in minutes
- events_in_session    : COUNT(*) of events in this session
- modules_touched      : COUNT(DISTINCT module_id) — how many modules the student visited
- primary_activity     : the event_type with the highest count in this session
                         (MODE — most frequent event type)
- is_long_session_flag : true if session_duration_mins > 480 (8 hours)

One row per session_id in the output.

In [0]:
# ============================================================
# CELL 10 — Compute primary_activity per session
#           = the event_type that appears most often in the session
#
# Spark has no built-in MODE aggregate.
# Approach:
#   1. COUNT event_type per (session_id, event_type)
#   2. Use ROW_NUMBER() OVER (PARTITION BY session_id ORDER BY cnt DESC)
#   3. Pick the row with rank=1 → that event_type is the primary activity
# ============================================================

# Step 1: count each event_type per session
event_type_counts_df = (
    lms_with_session_df
    .groupBy("session_id", "event_type")
    .agg(F.count("*").alias("event_type_cnt"))
)

# Step 2: rank within each session — highest count first
rank_window = Window.partitionBy("session_id").orderBy(F.desc("event_type_cnt"))

primary_activity_df = (
    event_type_counts_df
    .withColumn("rnk", F.row_number().over(rank_window))
    .filter(F.col("rnk") == 1)
    .select(
        F.col("session_id"),
        F.col("event_type").alias("primary_activity")
    )
)

print("── primary_activity distribution across all sessions ──")
primary_activity_df.groupBy("primary_activity").count().orderBy(F.desc("count")).show()

In [0]:
# ============================================================
# CELL 11 — GROUP BY session_id to produce one row per session
#           Joins primary_activity computed in Cell 10
# ============================================================

# Aggregate per session
session_agg_df = (
    lms_with_session_df
    .groupBy("session_id", "student_id")
    .agg(
        # First course_id seen in the session (most sessions are single-course)
        F.first("course_id", ignorenulls=True).alias("course_id"),
        # Session time boundaries
        F.min("event_ts").alias("session_start_ts"),
        F.max("event_ts").alias("session_end_ts"),
        # Total events in session
        F.count("*").alias("events_in_session"),
        # Distinct modules visited
        F.countDistinct("module_id").alias("modules_touched"),
        # Carry forward source file for lineage
        F.first("_source_file").alias("_source_file"),
        F.first("_load_ts").alias("bronze_load_ts"),
    )
)

# Compute session_duration_mins from start/end timestamps
session_with_duration_df = (
    session_agg_df
    .withColumn(
        "session_duration_mins",
        F.round(
            (F.unix_timestamp("session_end_ts") - F.unix_timestamp("session_start_ts"))
            / 60.0,
            2
        )
    )
    # Flag sessions > 8 hours — keep them, just mark the flag
    .withColumn(
        "is_long_session_flag",
        F.when(
            F.col("session_duration_mins") > LONG_SESSION_MINUTES,
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

# Join primary_activity
sessions_full_df = session_with_duration_df.join(
    primary_activity_df,
    on="session_id",
    how="left"
)

session_count = sessions_full_df.count()
long_sessions = sessions_full_df.filter(F.col("is_long_session_flag") == True).count()

print("── Session aggregation complete ──────────────────")
print(f"   Total sessions          : {session_count:,}")
print(f"   Long sessions (>8h)     : {long_sessions}  (flagged, not dropped)")
print()

print("── session_duration_mins stats ───────────────────")
sessions_full_df.select(
    F.min("session_duration_mins").alias("min_mins"),
    F.max("session_duration_mins").alias("max_mins"),
    F.round(F.avg("session_duration_mins"), 2).alias("avg_mins"),
    F.percentile_approx("session_duration_mins", 0.5).alias("median_mins")
).show()

print("── events_in_session stats ───────────────────────")
sessions_full_df.select(
    F.min("events_in_session").alias("min_events"),
    F.max("events_in_session").alias("max_events"),
    F.round(F.avg("events_in_session"), 2).alias("avg_events")
).show()

In [0]:
# ============================================================
# CELL 12 — Assemble final Silver DataFrame
#           Select all columns in clean order
#           Add Silver metadata columns
# ============================================================

learning_sessions_df = (
    sessions_full_df
    .select(
        # Identity
        "session_id",
        "student_id",
        "course_id",
        # Session boundaries
        "session_start_ts",
        "session_end_ts",
        # Computed metrics
        "session_duration_mins",
        "events_in_session",
        "modules_touched",
        "primary_activity",
        # Data quality flag
        "is_long_session_flag",
        # Bronze lineage
        "_source_file",
        "bronze_load_ts",
    )
    # Add Silver metadata
    .withColumn("_silver_load_ts", F.current_timestamp())
    .withColumn("_schema_version",  F.lit(SCHEMA_VERSION))
)

print("── Final Silver DataFrame ────────────────────────")
print(f"   Rows    : {learning_sessions_df.count():,}")
print(f"   Columns : {len(learning_sessions_df.columns)}")
print()
learning_sessions_df.printSchema()

In [0]:
# ============================================================
# CELL 13 — Write to silver.learning_sessions
#           Mode: OVERWRITE — sessions are fully recomputed from
#           all LMS events on every run
#           overwriteSchema=true allows schema evolution
# ============================================================

(
    learning_sessions_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SESSIONS_SILVER_TABLE)
)

final_count = spark.table(SESSIONS_SILVER_TABLE).count()
print(f"✅ Written to: {SESSIONS_SILVER_TABLE}")
print(f"   Rows in table: {final_count:,}")

In [0]:
# ============================================================
# CELL 14 — Verify silver.learning_sessions
# ============================================================

sess_df = spark.table(SESSIONS_SILVER_TABLE)

print("── learning_sessions ─────────────────────────────")
print(f"Total rows   : {sess_df.count():,}")
print(f"Columns      : {len(sess_df.columns)}")
print()

print("── primary_activity distribution ─────────────────")
sess_df.groupBy("primary_activity").count().orderBy(F.desc("count")).show(truncate=False)

print("── long session breakdown ────────────────────────")
sess_df.groupBy("is_long_session_flag").count().show()

print("── sessions per student (top 5) ──────────────────")
sess_df.groupBy("student_id") \
    .agg(F.count("session_id").alias("num_sessions")) \
    .orderBy(F.desc("num_sessions")) \
    .show(5, truncate=False)

print("── sample rows ───────────────────────────────────")
sess_df.select(
    "session_id", "student_id", "course_id",
    "session_duration_mins", "events_in_session",
    "modules_touched", "primary_activity", "is_long_session_flag"
).show(5, truncate=False)

## Part D — Data Quality Checks (Silver)

DQ checks for learning_sessions:
1. Sessions > 8 hours must have is_long_session_flag = true (audit log entry)
2. student_id and course_id must not be NULL
3. session_duration_mins must be >= 0 (negative duration = data error)
4. session_id must be unique (no duplicate sessions)

In [0]:
# ============================================================
# CELL 16 — DQ Check 1: sessions > 8 hours
#           These are data errors per business rules
#           Flag = true on the row AND write to dq_audit
#           NOT dropped — kept with flag for investigation
# ============================================================

sess_df = spark.table(SESSIONS_SILVER_TABLE)

long_sessions = (
    sess_df
    .filter(F.col("is_long_session_flag") == True)
    .withColumn("dq_check_name", F.lit("session_exceeds_8_hours"))
    .withColumn("dq_table",      F.lit(SESSIONS_SILVER_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("session_duration_mins exceeds 480:"),
        F.col("session_duration_mins").cast("string"),
        F.lit("session_id:"), F.col("session_id")
    ))
)

long_count = long_sessions.count()
print("── DQ Check 1: sessions > 8 hours ────────────────")
print(f"   Long sessions flagged: {long_count}")

if long_count > 0:
    long_sessions.select(
        "session_id", "student_id", "session_duration_mins"
    ).orderBy(F.desc("session_duration_mins")).show(5, truncate=False)

    (
        long_sessions
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "session_id", "student_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {long_count} sessions logged to {DQ_AUDIT_TABLE} (rows kept with flag=true)")
else:
    print("   ✅ No sessions exceed 8 hours in this dataset")

In [0]:
# ============================================================
# CELL 17 — DQ Check 2: NULL student_id and course_id
#           student_id is the primary identity field for sessions
#           course_id links sessions to downstream Silver tables
# ============================================================

sess_df = spark.table(SESSIONS_SILVER_TABLE)
total   = sess_df.count()

for col_name in ["student_id", "course_id"]:
    null_rows  = sess_df.filter(F.col(col_name).isNull())
    null_count = null_rows.count()
    pct        = (null_count / total * 100) if total > 0 else 0
    status     = "✅" if null_count == 0 else "⚠"
    alert      = "  ← ALERT: exceeds 0.5% threshold!" if pct > 0.5 else ""

    print(f"── DQ Check 2: NULL {col_name} ─────────────────")
    print(f"   {status}  nulls={null_count:,}  ({pct:.3f}%){alert}")

    if null_count > 0:
        flagged = (
            null_rows
            .withColumn("dq_check_name", F.lit(f"null_{col_name}_sessions"))
            .withColumn("dq_table",      F.lit(SESSIONS_SILVER_TABLE))
            .withColumn("dq_severity",   F.lit("HIGH"))
            .withColumn("dq_checked_at", F.current_timestamp())
            .withColumn("dq_message",    F.lit(f"{col_name} is NULL in learning_sessions"))
        )
        (
            flagged
            .select("dq_check_name", "dq_table", "dq_severity",
                    "dq_checked_at", "dq_message", "session_id")
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(DQ_AUDIT_TABLE)
        )
        print(f"   ⚠ {null_count} rows flagged → written to {DQ_AUDIT_TABLE}")
    print()

In [0]:
# ============================================================
# CELL 18 — DQ Check 3a: session_duration_mins must be >= 0
#           A negative duration means event_ts ordering was wrong
#           or the timestamps are corrupted
#
#           DQ Check 3b: session_id must be unique
#           Duplicate session_ids would cause fan-out in joins
# ============================================================

sess_df = spark.table(SESSIONS_SILVER_TABLE)

# ── 3a: Negative duration ──────────────────────────────────
negative_duration = (
    sess_df
    .filter(F.col("session_duration_mins") < 0)
    .withColumn("dq_check_name", F.lit("negative_session_duration"))
    .withColumn("dq_table",      F.lit(SESSIONS_SILVER_TABLE))
    .withColumn("dq_severity",   F.lit("HIGH"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("session_duration_mins is negative:"),
        F.col("session_duration_mins").cast("string"),
        F.lit("session_id:"), F.col("session_id")
    ))
)

neg_count = negative_duration.count()
print("── DQ Check 3a: negative session_duration_mins ───")
if neg_count > 0:
    (
        negative_duration
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "session_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {neg_count} sessions with negative duration → flagged in {DQ_AUDIT_TABLE}")
    negative_duration.select(
        "session_id", "session_duration_mins"
    ).show(5, truncate=False)
else:
    print("   ✅ PASSED — all session_duration_mins >= 0")

print()

# ── 3b: Duplicate session_ids ──────────────────────────────
duplicate_sessions = (
    sess_df
    .groupBy("session_id")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)

dup_count = duplicate_sessions.count()
print("── DQ Check 3b: duplicate session_ids ───────────")
if dup_count > 0:
    print(f"   ⚠ {dup_count} duplicate session_ids found:")
    duplicate_sessions.show(5, truncate=False)
else:
    print("   ✅ PASSED — all session_ids are unique")

In [0]:
%sql
-- ============================================================
-- CELL 19 — SQL verification of silver.learning_sessions
-- ============================================================

SELECT
    COUNT(*)                                          AS total_sessions,
    COUNT(DISTINCT student_id)                        AS unique_students,
    COUNT(DISTINCT course_id)                         AS unique_courses,
    ROUND(AVG(session_duration_mins), 2)              AS avg_duration_mins,
    MAX(session_duration_mins)                        AS max_duration_mins,
    ROUND(AVG(events_in_session), 2)                  AS avg_events_per_session,
    ROUND(AVG(modules_touched), 2)                    AS avg_modules_touched,
    SUM(CASE WHEN is_long_session_flag = true THEN 1 ELSE 0 END) AS long_sessions_flagged,
    SUM(CASE WHEN course_id IS NULL THEN 1 ELSE 0 END) AS null_course_id_count
FROM mini_project_grp6.silver.learning_sessions;

In [0]:
%sql
-- ============================================================
-- CELL 20 — Sessions per student distribution
--           Useful sanity check: every student should have
--           at least 1 session if they appear in LMS events
-- ============================================================

SELECT
    num_sessions,
    COUNT(*) AS student_count
FROM (
    SELECT
        student_id,
        COUNT(session_id) AS num_sessions
    FROM mini_project_grp6.silver.learning_sessions
    GROUP BY student_id
)
GROUP BY num_sessions
ORDER BY num_sessions;

In [0]:
# ============================================================
# CELL 21 — Task 2.2 completion summary
# ============================================================

final_df    = spark.table(SESSIONS_SILVER_TABLE)
final_count = final_df.count()
long_count  = final_df.filter(F.col("is_long_session_flag") == True).count()
students    = final_df.select("student_id").distinct().count()

print("═" * 58)
print("  TASK 2.2 COMPLETE — Silver Learning Sessions")
print("═" * 58)
print()
print(f"  ✅ {SESSIONS_SILVER_TABLE}")
print(f"      Total sessions    : {final_count:,}")
print(f"      Unique students   : {students:,}")
print(f"      Long sessions (>8h): {long_count}  (flagged, not dropped)")
print(f"      Mode              : OVERWRITE (fully recomputed)")
print()
print("  Session detection logic:")
print(f"      Gap threshold     : {SESSION_GAP_MINUTES} minutes")
print(f"      Method            : LAG(event_ts) → gap_seconds → is_new_session")
print(f"      session_id        : student_id + '_S_' + cumulative SUM(is_new_session)")
print()
print("  Computed columns:")
print("      session_duration_mins  — MAX(event_ts) - MIN(event_ts) in session")
print("      events_in_session      — COUNT(*) per session_id")
print("      modules_touched        — COUNT(DISTINCT module_id) per session_id")
print("      primary_activity       — MODE(event_type) per session_id")
print("      is_long_session_flag   — session_duration_mins > 480")
print()
print("  DQ checks run:")
print("      Cell 16 — sessions > 8h flagged + logged to dq_audit")
print("      Cell 17 — NULL student_id / course_id")
print("      Cell 18 — negative duration + duplicate session_ids")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: Task 2.3 → 06_silver_student_engagement")
print("         Multi-source aggregate: video + quiz + discussion + logins")
print("═" * 58)